In [1]:
# HW1 Q1 - 2-head word 빈도
# 환경 준비: PySpark 설치, SparkContext 생성
!pip -q install pyspark
from pyspark import SparkContext
sc = SparkContext.getOrCreate()

In [2]:
# 데이터 내려받기 (textFile은 https 주소를 직접 못 읽으므로 파일로 받음, 내용은 손대지 않음)
!wget -q -O sherlock.txt https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/quiz/sherlock.txt
!head -3 sherlock.txt

A STUDY IN SCARLET.




In [3]:
# 0. 읽기: 원소 = 줄 하나(문자열), 빈 줄도 '' 원소 하나
rdd = sc.textFile("sherlock.txt")

# 1. 줄 -> 단어: lambda가 리스트 하나를 돌려줌 -> flatMap이 뜯어서 단어마다 원소로
words = rdd.flatMap(lambda line: line.split(' '))

# 2. 길이 > 2인 단어만 (반드시 자르기 전에). '', 'A', 'IN', 'I.' 등은 빠짐
long_words = words.filter(lambda w: len(w) > 2)

# 3. 소문자로 바꾼 뒤 앞 두 글자, (head, 1) 쌍. 원소 하나 -> 원소 하나라서 map
pairs = long_words.map(lambda w: (w.lower()[:2], 1))

# 4. head별 빈도. lambda는 같은 키의 값 둘만 받음 (키는 Spark가 들고 있음). 셔플 발생(wide)
counts = pairs.reduceByKey(lambda x, y: x + y)

# 5. 빈도 < 32인 것만. filter의 lambda는 튜플 통째로 받으므로 kv[1]이 빈도
small = counts.filter(lambda kv: kv[1] < 32)

# 6. 빈도만 꺼내(map) 모두 더함(reduce). reduce는 action -> 여기서 위 계획 전체가 실행됨(lazy)
total = small.map(lambda kv: kv[1]).reduce(lambda a, b: a + b)

In [4]:
print(total)

772
